<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/Day_18_Measuring_and_detecting_LLM_hallucinations_with_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Day 16: Understanding and Detecting AI Hallucinations

## Objective

The goal of this project is to evaluate and compare hallucinations produced by an LLM under two different conditions:

1. **Without context or retrieval**
2. **With Retrieval-Augmented Generation (RAG)**

## Experiment Setup

- Created a dataset of **20 factual questions**
- Covered multiple knowledge domains to test breadth
- Defined ground-truth answers for each question
- Used **GPT-4o-mini** to generate responses without retrieval context
- Recorded all responses in a structured format
- Manually evaluated each response using predefined scoring criteria
- Classified responses as:
  - Correct
  - Partially Correct
  - Hallucinated

## Hallucination Categories

Hallucinated responses were categorized into:

- **Fabricated specific facts**
- **Outdated information presented as current**
- **Confident wrong answers**
- **Plausible-sounding but unverifiable claims**

## RAG Comparison

The same 20 questions were then passed through the RAG pipeline.

Responses from both conditions were evaluated and compared to calculate:

- Total hallucinations
- Hallucination rate without retrieval
- Hallucination rate with RAG
- Difference in hallucination rates

## Key Goal

This experiment helps evaluate whether providing retrieved context can ground LLM responses and reduce the likelihood of hallucinations.

> **Key Insight:** Building reliable AI systems is not just about generating answers, but also about evaluating whether those answers are factual, grounded, and trustworthy.

In [ ]:
# Hallucination Measurement Experiment (No OpenAI — 100% Free / Colab)

Quantifies how often and why an LLM fabricates information, comparing a
**no-context** condition against a **RAG-grounded** condition, then runs a
four-signal rule-based hallucination detector.

Everything here runs on free, open-source models:
- **Generation**: `Qwen/Qwen2.5-1.5B-Instruct` (small instruct model, runs on Colab's free T4 GPU or CPU)
- **Retrieval embeddings**: `sentence-transformers/all-MiniLM-L6-v2`

**Before running:** In Colab, go to `Runtime > Change runtime type > T4 GPU` for speed (CPU also works, just slower).


In [1]:
!pip install -q -U transformers accelerate sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 16.0 MB/s eta 0:00:00


In [ ]:
## 1. Load the generation model

Small open instruct model — no API key needed.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # ~3GB download, good quality/speed tradeoff for Colab

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    model = model.to(device)

def generate(user_prompt, system_prompt="You are a helpful assistant. Answer directly and concisely.", max_new_tokens=150):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # deterministic/greedy -> reproducible for measurement
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print(generate("What is the capital of France?"))


Using device: cpu


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Paris.


In [3]:
dataset = [
    {"id": 1, "domain": "science", "question": "What is the chemical symbol for gold?", "ground_truth": "Au"},
    {"id": 2, "domain": "science", "question": "What is the approximate speed of light in a vacuum, in km/s?", "ground_truth": "About 299,792 km/s (roughly 300,000 km/s)"},
    {"id": 3, "domain": "science", "question": "What organelle is known as the powerhouse of the cell?", "ground_truth": "The mitochondrion"},
    {"id": 4, "domain": "science", "question": "What element has atomic number 1?", "ground_truth": "Hydrogen"},
    {"id": 5, "domain": "science", "question": "What is the boiling point of water at sea level in Celsius?", "ground_truth": "100°C"},
    {"id": 6, "domain": "science", "question": "Who developed the theory of general relativity?", "ground_truth": "Albert Einstein"},
    {"id": 7, "domain": "science", "question": "What gas do plants primarily absorb from the atmosphere for photosynthesis?", "ground_truth": "Carbon dioxide (CO2)"},
    {"id": 8, "domain": "history", "question": "In what year did World War II end?", "ground_truth": "1945"},
    {"id": 9, "domain": "history", "question": "Who was the first President of the United States?", "ground_truth": "George Washington"},
    {"id": 10, "domain": "history", "question": "In what year did the Berlin Wall fall?", "ground_truth": "1989"},
    {"id": 11, "domain": "history", "question": "Who was the principal author of the Declaration of Independence?", "ground_truth": "Thomas Jefferson"},
    {"id": 12, "domain": "history", "question": "In what year did India gain independence from British rule?", "ground_truth": "1947"},
    {"id": 13, "domain": "history", "question": "Who was the first person to walk on the Moon?", "ground_truth": "Neil Armstrong"},
    {"id": 14, "domain": "history", "question": "In what year did the French Revolution begin?", "ground_truth": "1789"},
    {"id": 15, "domain": "geography", "question": "What is the capital of Australia?", "ground_truth": "Canberra"},
    {"id": 16, "domain": "geography", "question": "What is generally considered the longest river in the world?", "ground_truth": "The Nile (commonly cited; the Amazon is a debated alternative)"},
    {"id": 17, "domain": "geography", "question": "What is the smallest country in the world by area?", "ground_truth": "Vatican City"},
    {"id": 18, "domain": "geography", "question": "What is the tallest mountain above sea level?", "ground_truth": "Mount Everest"},
    {"id": 19, "domain": "geography", "question": "What is the capital of Canada?", "ground_truth": "Ottawa"},
    {"id": 20, "domain": "geography", "question": "What is the largest hot desert in the world?", "ground_truth": "The Sahara Desert"},
]
print(f"{len(dataset)} questions loaded across domains: {sorted(set(q['domain'] for q in dataset))}")


20 questions loaded across domains: ['geography', 'history', 'science']


In [4]:
results_no_context = {}
for item in dataset:
    qid = str(item["id"])
    response = generate(item["question"])
    results_no_context[qid] = {
        "question": item["question"],
        "domain": item["domain"],
        "ground_truth": item["ground_truth"],
        "response": response,
    }
    print(f"[{qid}/20] {item['question']}\n  -> {response}\n")

import json
with open("results_no_context.json", "w") as f:
    json.dump(results_no_context, f, indent=2)
print("Saved results_no_context.json")


[1/20] What is the chemical symbol for gold?
  -> The chemical symbol for gold is Au.

[2/20] What is the approximate speed of light in a vacuum, in km/s?
  -> The approximate speed of light in a vacuum is 299,792 kilometers per second (km/s).

[3/20] What organelle is known as the powerhouse of the cell?
  -> The organelle known as the powerhouse of the cell is the mitochondria.

[4/20] What element has atomic number 1?
  -> Hydrogen has atomic number 1.

[5/20] What is the boiling point of water at sea level in Celsius?
  -> The boiling point of water at sea level is 100 degrees Celsius (212 degrees Fahrenheit).

[6/20] Who developed the theory of general relativity?
  -> Albert Einstein developed the theory of general relativity.

[7/20] What gas do plants primarily absorb from the atmosphere for photosynthesis?
  -> Plants primarily absorb carbon dioxide (CO2) from the atmosphere for photosynthesis.

[8/20] In what year did World War II end?
  -> World War II ended on September 2, 

In [5]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)
corpus_embeddings = embedder.encode(CORPUS, convert_to_tensor=True)

TOP_K = 3

def retrieve(query, k=TOP_K):
    query_embedding = embedder.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=k)[0]
    return [CORPUS[hit["corpus_id"]] for hit in hits]

RAG_SYSTEM_PROMPT = (
    "You are a factual assistant. Answer the user's question using ONLY the "
    "information in the provided context. If the context does not contain the "
    "answer, say 'The provided context does not contain this information' "
    "instead of guessing. Do not use outside knowledge."
)

def answer_with_rag(question):
    chunks = retrieve(question)
    context_block = "\n".join(f"- {c}" for c in chunks)
    user_prompt = f"Context:\n{context_block}\n\nQuestion: {question}"
    answer = generate(user_prompt, system_prompt=RAG_SYSTEM_PROMPT, max_new_tokens=150)
    return answer, chunks

# quick smoke test
a, c = answer_with_rag("What is the capital of Australia?")
print(a)
print(c)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

NameError: name 'CORPUS' is not defined

In [6]:
CORPUS = [
    "Gold has the chemical symbol Au, derived from the Latin word 'aurum'. It is a dense, soft, yellow metal.",
    "The speed of light in a vacuum is exactly 299,792,458 metres per second, approximately 299,792 kilometres per second.",
    "The mitochondrion is an organelle found in most eukaryotic cells, often called the powerhouse of the cell because it generates ATP.",
    "Hydrogen is the chemical element with atomic number 1 and the symbol H. It is the lightest and most abundant element in the universe.",
    "At standard atmospheric pressure (sea level), water boils at 100 degrees Celsius (212 degrees Fahrenheit).",
    "Albert Einstein published the theory of general relativity in 1915, expanding on his earlier special relativity from 1905.",
    "During photosynthesis, plants absorb carbon dioxide from the atmosphere and release oxygen as a byproduct.",
    "World War II ended in 1945, with Germany surrendering in May and Japan surrendering in September following the atomic bombings.",
    "George Washington served as the first President of the United States from 1789 to 1797.",
    "The Berlin Wall fell on November 9, 1989, marking a major symbolic moment in the end of the Cold War.",
    "Thomas Jefferson was the principal author of the United States Declaration of Independence, adopted in 1776.",
    "India gained independence from British colonial rule on August 15, 1947.",
    "Neil Armstrong became the first person to walk on the Moon on July 20, 1969, during the Apollo 11 mission.",
    "The French Revolution began in 1789 with the storming of the Bastille and led to major political upheaval in France.",
    "Canberra is the capital city of Australia, chosen as a compromise between the rival cities of Sydney and Melbourne.",
    "The Nile River in Africa is traditionally cited as the longest river in the world at roughly 6,650 kilometres, though some studies argue the Amazon is longer.",
    "Vatican City is the smallest sovereign state in the world by both area and population, an enclave within Rome.",
    "Mount Everest, located in the Himalayas on the border of Nepal and Tibet, is the tallest mountain above sea level at 8,849 metres.",
    "Ottawa is the capital city of Canada, located in the province of Ontario.",
    "The Sahara is the largest hot desert in the world, covering much of North Africa.",
]
print(f"Corpus size: {len(CORPUS)} passages")

from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)
corpus_embeddings = embedder.encode(CORPUS, convert_to_tensor=True)

TOP_K = 3

def retrieve(query, k=TOP_K):
    query_embedding = embedder.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=k)[0]
    return [CORPUS[hit["corpus_id"]] for hit in hits]

RAG_SYSTEM_PROMPT = (
    "You are a factual assistant. Answer the user's question using ONLY the "
    "information in the provided context. If the context does not contain the "
    "answer, say 'The provided context does not contain this information' "
    "instead of guessing. Do not use outside knowledge."
)

def answer_with_rag(question):
    chunks = retrieve(question)
    context_block = "\n".join(f"- {c}" for c in chunks)
    user_prompt = f"Context:\n{context_block}\n\nQuestion: {question}"
    answer = generate(user_prompt, system_prompt=RAG_SYSTEM_PROMPT, max_new_tokens=150)
    return answer, chunks

# quick smoke test
a, c = answer_with_rag("What is the capital of Australia?")
print(a)
print(c)

Corpus size: 20 passages


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Canberra is the capital of Australia.
['Canberra is the capital city of Australia, chosen as a compromise between the rival cities of Sydney and Melbourne.', 'Ottawa is the capital city of Canada, located in the province of Ontario.', "Gold has the chemical symbol Au, derived from the Latin word 'aurum'. It is a dense, soft, yellow metal."]


In [8]:
results_rag = {}
for item in dataset:
    qid = str(item["id"])
    response, chunks = answer_with_rag(item["question"])
    results_rag[qid] = {
        "question": item["question"],
        "domain": item["domain"],
        "ground_truth": item["ground_truth"],
        "response": response,
        "retrieved_chunks": chunks,
    }
    print(f"[{qid}/20] {item['question']}\n  -> {response}\n")

with open("results_rag.json", "w") as f:
    json.dump(results_rag, f, indent=2)
print("Saved results_rag.json")


[1/20] What is the chemical symbol for gold?
  -> The chemical symbol for gold is Au.

[2/20] What is the approximate speed of light in a vacuum, in km/s?
  -> The approximate speed of light in a vacuum is 299,792 kilometers per second.

[3/20] What organelle is known as the powerhouse of the cell?
  -> The organelle known as the powerhouse of the cell is the mitochondrion.

[4/20] What element has atomic number 1?
  -> Hydrogen

[5/20] What is the boiling point of water at sea level in Celsius?
  -> At standard atmospheric pressure (sea level), water boils at 100 degrees Celsius (212 degrees Fahrenheit).

[6/20] Who developed the theory of general relativity?
  -> Albert Einstein developed the theory of general relativity.

[7/20] What gas do plants primarily absorb from the atmosphere for photosynthesis?
  -> Plants primarily absorb carbon dioxide (CO2) from the atmosphere for photosynthesis.

[8/20] In what year did World War II end?
  -> World War II ended in 1945.

[9/20] Who was 

In [9]:
LABELS = ["correct", "partially_correct", "hallucinated", "abstained"]
HALLUCINATION_TYPES = [
    "fabricated_specific_fact",
    "outdated_info_as_current",
    "confident_wrong_answer",
    "plausible_unverifiable_claim",
]

def prompt_choice(prompt_text, options):
    while True:
        print(prompt_text)
        for i, opt in enumerate(options, 1):
            print(f"  {i}. {opt}")
        raw = input("> ").strip()
        if raw.isdigit() and 1 <= int(raw) <= len(options):
            return options[int(raw) - 1]
        print("Invalid choice, try again.\n")

def manual_eval(results, out_path):
    scores = {}
    for qid, item in results.items():
        print("=" * 70)
        print(f"Q{qid}: {item['question']}")
        print(f"Ground truth: {item['ground_truth']}")
        print(f"Response:\n{item['response']}")
        print("=" * 70)
        label = prompt_choice("Classification:", LABELS)
        entry = {"label": label}
        if label == "hallucinated":
            entry["hallucination_type"] = prompt_choice("Hallucination type:", HALLUCINATION_TYPES)
        notes = input("Optional notes (enter to skip): ").strip()
        if notes:
            entry["notes"] = notes
        scores[qid] = entry
        with open(out_path, "w") as f:
            json.dump(scores, f, indent=2)
    print(f"Saved {out_path}")
    return scores

In [10]:
from collections import Counter

def analyze(scores):
    n = len(scores)
    label_counts = Counter(v["label"] for v in scores.values())
    hallucinated = [v for v in scores.values() if v["label"] == "hallucinated"]
    type_counts = Counter(v.get("hallucination_type", "unspecified") for v in hallucinated)
    rate = round(100 * label_counts.get("hallucinated", 0) / n, 1) if n else 0.0
    return {"n": n, "label_counts": dict(label_counts), "hallucination_rate_pct": rate,
            "hallucination_type_counts": dict(type_counts)}

def print_table(name, stats):
    print(f"\n=== {name} ===")
    print(f"N = {stats['n']}")
    for label in LABELS:
        count = stats["label_counts"].get(label, 0)
        pct = round(100 * count / stats["n"], 1) if stats["n"] else 0
        print(f"  {label:20s}: {count:2d}  ({pct}%)")
    print(f"  Hallucination rate  : {stats['hallucination_rate_pct']}%")
    if stats["hallucination_type_counts"]:
        print("  Failure breakdown:")
        for t in HALLUCINATION_TYPES:
            c = stats["hallucination_type_counts"].get(t, 0)
            if c:
                print(f"    - {t:32s}: {c}")

no_ctx_stats = analyze(manual_scores_no_context)
rag_stats = analyze(manual_scores_rag)
print_table("NO CONTEXT", no_ctx_stats)
print_table("RAG", rag_stats)

diff = round(no_ctx_stats["hallucination_rate_pct"] - rag_stats["hallucination_rate_pct"], 1)
print("\n=== COMPARISON ===")
print(f"No-context hallucination rate: {no_ctx_stats['hallucination_rate_pct']}%")
print(f"RAG hallucination rate       : {rag_stats['hallucination_rate_pct']}%")
print(f"Absolute reduction from RAG  : {diff} percentage points")

with open("rate_comparison.json", "w") as f:
    json.dump({"no_context": no_ctx_stats, "rag": rag_stats, "reduction_pp": diff}, f, indent=2)
print("Saved rate_comparison.json")


NameError: name 'manual_scores_no_context' is not defined

In [11]:


# Score the NO-CONTEXT responses
manual_scores_no_context = manual_eval(results_no_context, "manual_scores_no_context.json")

Q1: What is the chemical symbol for gold?
Ground truth: Au
Response:
The chemical symbol for gold is Au.
Classification:
  1. correct
  2. partially_correct
  3. hallucinated
  4. abstained
> 5.precise
Invalid choice, try again.

Classification:
  1. correct
  2. partially_correct
  3. hallucinated
  4. abstained
> Au
Invalid choice, try again.

Classification:
  1. correct
  2. partially_correct
  3. hallucinated
  4. abstained
> 3
Hallucination type:
  1. fabricated_specific_fact
  2. outdated_info_as_current
  3. confident_wrong_answer
  4. plausible_unverifiable_claim
> 1
Optional notes (enter to skip): 
Q2: What is the approximate speed of light in a vacuum, in km/s?
Ground truth: About 299,792 km/s (roughly 300,000 km/s)
Response:
The approximate speed of light in a vacuum is 299,792 kilometers per second (km/s).
Classification:
  1. correct
  2. partially_correct
  3. hallucinated
  4. abstained
> 3
Hallucination type:
  1. fabricated_specific_fact
  2. outdated_info_as_current


In [12]:
# Score the RAG responses
manual_scores_rag = manual_eval(results_rag, "manual_scores_rag.json")

Q1: What is the chemical symbol for gold?
Ground truth: Au
Response:
The chemical symbol for gold is Au.
Classification:
  1. correct
  2. partially_correct
  3. hallucinated
  4. abstained
> 3
Hallucination type:
  1. fabricated_specific_fact
  2. outdated_info_as_current
  3. confident_wrong_answer
  4. plausible_unverifiable_claim
> 2
Optional notes (enter to skip): 3
Q2: What is the approximate speed of light in a vacuum, in km/s?
Ground truth: About 299,792 km/s (roughly 300,000 km/s)
Response:
The approximate speed of light in a vacuum is 299,792 kilometers per second.
Classification:
  1. correct
  2. partially_correct
  3. hallucinated
  4. abstained
> 2
Optional notes (enter to skip): 3
Q3: What organelle is known as the powerhouse of the cell?
Ground truth: The mitochondrion
Response:
The organelle known as the powerhouse of the cell is the mitochondrion.
Classification:
  1. correct
  2. partially_correct
  3. hallucinated
  4. abstained
> 4
Optional notes (enter to skip): 


In [13]:
from collections import Counter

def analyze(scores):
    n = len(scores)
    label_counts = Counter(v["label"] for v in scores.values())
    hallucinated = [v for v in scores.values() if v["label"] == "hallucinated"]
    type_counts = Counter(v.get("hallucination_type", "unspecified") for v in hallucinated)
    rate = round(100 * label_counts.get("hallucinated", 0) / n, 1) if n else 0.0
    return {"n": n, "label_counts": dict(label_counts), "hallucination_rate_pct": rate,
            "hallucination_type_counts": dict(type_counts)}

def print_table(name, stats):
    print(f"\n=== {name} ===")
    print(f"N = {stats['n']}")
    for label in LABELS:
        count = stats["label_counts"].get(label, 0)
        pct = round(100 * count / stats["n"], 1) if stats["n"] else 0
        print(f"  {label:20s}: {count:2d}  ({pct}%)")
    print(f"  Hallucination rate  : {stats['hallucination_rate_pct']}%")
    if stats["hallucination_type_counts"]:
        print("  Failure breakdown:")
        for t in HALLUCINATION_TYPES:
            c = stats["hallucination_type_counts"].get(t, 0)
            if c:
                print(f"    - {t:32s}: {c}")

no_ctx_stats = analyze(manual_scores_no_context)
rag_stats = analyze(manual_scores_rag)
print_table("NO CONTEXT", no_ctx_stats)
print_table("RAG", rag_stats)

diff = round(no_ctx_stats["hallucination_rate_pct"] - rag_stats["hallucination_rate_pct"], 1)
print("\n=== COMPARISON ===")
print(f"No-context hallucination rate: {no_ctx_stats['hallucination_rate_pct']}%")
print(f"RAG hallucination rate       : {rag_stats['hallucination_rate_pct']}%")
print(f"Absolute reduction from RAG  : {diff} percentage points")

with open("rate_comparison.json", "w") as f:
    json.dump({"no_context": no_ctx_stats, "rag": rag_stats, "reduction_pp": diff}, f, indent=2)
print("Saved rate_comparison.json")


=== NO CONTEXT ===
N = 20
  correct             :  0  (0.0%)
  partially_correct   :  0  (0.0%)
  hallucinated        : 20  (100.0%)
  abstained           :  0  (0.0%)
  Hallucination rate  : 100.0%
  Failure breakdown:
    - fabricated_specific_fact        : 5
    - outdated_info_as_current        : 6
    - confident_wrong_answer          : 3
    - plausible_unverifiable_claim    : 6

=== RAG ===
N = 20
  correct             :  0  (0.0%)
  partially_correct   :  1  (5.0%)
  hallucinated        : 18  (90.0%)
  abstained           :  1  (5.0%)
  Hallucination rate  : 90.0%
  Failure breakdown:
    - fabricated_specific_fact        : 4
    - outdated_info_as_current        : 7
    - confident_wrong_answer          : 2
    - plausible_unverifiable_claim    : 5

=== COMPARISON ===
No-context hallucination rate: 100.0%
RAG hallucination rate       : 90.0%
Absolute reduction from RAG  : 10.0 percentage points
Saved rate_comparison.json


In [14]:
import re

STOPWORDS = set(
    "a an the is are was were be been being of to in on at for and or but "
    "with as by from this that it its into about over under than then "
    "so if not no do does did has have had will would can could should "
    "i you he she they we my your his her their our".split()
)

NUM_RE = re.compile(r"\b\d[\d,\.]*\b")
YEAR_RE = re.compile(r"\b(1[0-9]{3}|20[0-9]{2})\b")
ENTITY_RE = re.compile(r"\b([A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+)*)\b")

def extract_claims(text):
    numbers = set(NUM_RE.findall(text))
    years = set(YEAR_RE.findall(text))
    entities = set(m.strip() for m in ENTITY_RE.findall(text) if len(m.strip()) > 2)
    return numbers | years | entities

def unsupported_claim_check(response, context_text):
    claims = extract_claims(response)
    unsupported = [c for c in claims if c not in context_text]
    return {"flagged": len(unsupported) > 0, "unsupported_claims": unsupported, "total_claims": len(claims)}

def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    return set(w for w in words if w not in STOPWORDS)

def jaccard_overlap_check(response, context_text, threshold=0.15):
    resp_tokens, ctx_tokens = tokenize(response), tokenize(context_text)
    if not resp_tokens or not ctx_tokens:
        jaccard = 0.0
    else:
        jaccard = len(resp_tokens & ctx_tokens) / len(resp_tokens | ctx_tokens)
    return {"flagged": jaccard < threshold, "jaccard": round(jaccard, 3), "threshold": threshold}

def contradiction_check(response, context_text):
    # secondary LLM call using the SAME local model (no paid API needed)
    prompt = (
        f"Context:\n{context_text}\n\nResponse:\n{response}\n\n"
        "Does this response contradict the provided context? "
        "Answer with exactly one word: yes or no."
    )
    answer = generate(prompt, system_prompt="Answer with exactly one word: yes or no.", max_new_tokens=5).strip().lower()
    flagged = answer.startswith("yes")
    return {"flagged": flagged, "raw_answer": answer}

def has_factual_assertion(response):
    return len(extract_claims(response)) > 0

def citation_absence_check(response, chunks, min_shared_words=6):
    if not has_factual_assertion(response):
        return {"flagged": False, "reason": "no verifiable assertion to check"}
    resp_words = re.findall(r"[a-z0-9]+", response.lower())
    grounded = False
    for chunk in chunks:
        chunk_word_set = set(re.findall(r"[a-z0-9]+", chunk.lower()))
        run = 0
        for w in resp_words:
            if w in chunk_word_set and w not in STOPWORDS:
                run += 1
                if run >= min_shared_words:
                    grounded = True
                    break
            else:
                run = 0
        if grounded:
            break
    return {"flagged": not grounded, "grounded_in_source": grounded}

WEIGHTS = {"unsupported_claim": 0.35, "low_overlap": 0.20, "contradiction": 0.30, "no_citation": 0.15}

def score_response(question, response, chunks):
    context_text = " ".join(chunks)
    s1 = unsupported_claim_check(response, context_text)
    s2 = jaccard_overlap_check(response, context_text)
    s3 = contradiction_check(response, context_text)
    s4 = citation_absence_check(response, chunks)
    risk = (WEIGHTS["unsupported_claim"] * s1["flagged"] + WEIGHTS["low_overlap"] * s2["flagged"]
            + WEIGHTS["contradiction"] * s3["flagged"] + WEIGHTS["no_citation"] * s4["flagged"])
    return {
        "question": question, "response": response,
        "signals": {"1_unsupported_claims": s1, "2_retrieval_overlap": s2,
                    "3_contradiction": s3, "4_citation_absence": s4},
        "hallucination_risk_score": round(risk, 2),
        "flag_count": sum([s1["flagged"], s2["flagged"], s3["flagged"], s4["flagged"]]),
    }

def print_breakdown(result):
    print("=" * 70)
    print(f"Q: {result['question']}")
    print(f"Response: {result['response'][:200]}")
    print("-" * 70)
    s = result["signals"]
    print(f"  [1] Unsupported claims : {'FLAG' if s['1_unsupported_claims']['flagged'] else 'ok'}  ({s['1_unsupported_claims']['unsupported_claims']})")
    print(f"  [2] Jaccard overlap    : {'FLAG' if s['2_retrieval_overlap']['flagged'] else 'ok'}  (score={s['2_retrieval_overlap']['jaccard']}, threshold={s['2_retrieval_overlap']['threshold']})")
    print(f"  [3] Contradiction      : {'FLAG' if s['3_contradiction']['flagged'] else 'ok'}  (model said: {s['3_contradiction']['raw_answer']})")
    print(f"  [4] Citation absence   : {'FLAG' if s['4_citation_absence']['flagged'] else 'ok'}")
    print(f"  --> Hallucination risk score: {result['hallucination_risk_score']}  ({result['flag_count']}/4 signals flagged)")
    print("=" * 70 + "\n")

detector_results = {}
for qid, item in results_rag.items():
    result = score_response(item["question"], item["response"], item["retrieved_chunks"])
    print_breakdown(result)
    detector_results[qid] = result

avg_risk = round(sum(r["hallucination_risk_score"] for r in detector_results.values()) / len(detector_results), 3)
high_risk = [qid for qid, r in detector_results.items() if r["hallucination_risk_score"] >= 0.5]
print(f"Average hallucination risk score across {len(detector_results)} responses: {avg_risk}")
print(f"High-risk responses (score >= 0.5): {high_risk}")

with open("detector_results.json", "w") as f:
    json.dump(detector_results, f, indent=2)
print("Saved detector_results.json")


Q: What is the chemical symbol for gold?
Response: The chemical symbol for gold is Au.
----------------------------------------------------------------------
  [1] Unsupported claims : ok  ([])
  [2] Jaccard overlap    : FLAG  (score=0.118, threshold=0.15)
  [3] Contradiction      : ok  (model said: no)
  [4] Citation absence   : FLAG
  --> Hallucination risk score: 0.35  (2/4 signals flagged)

Q: What is the approximate speed of light in a vacuum, in km/s?
Response: The approximate speed of light in a vacuum is 299,792 kilometers per second.
----------------------------------------------------------------------
  [1] Unsupported claims : ok  ([])
  [2] Jaccard overlap    : ok  (score=0.189, threshold=0.15)
  [3] Contradiction      : ok  (model said: no)
  [4] Citation absence   : FLAG
  --> Hallucination risk score: 0.15  (1/4 signals flagged)

Q: What organelle is known as the powerhouse of the cell?
Response: The organelle known as the powerhouse of the cell is the mitochondrion.
--

In [15]:
import shutil
files_to_zip = ["results_no_context.json", "results_rag.json",
                 "manual_scores_no_context.json", "manual_scores_rag.json",
                 "rate_comparison.json", "detector_results.json"]
import os
os.makedirs("hallucination_experiment_outputs", exist_ok=True)
for f in files_to_zip:
    if os.path.exists(f):
        shutil.copy(f, f"hallucination_experiment_outputs/{f}")
shutil.make_archive("hallucination_experiment_outputs", "zip", "hallucination_experiment_outputs")

try:
    from google.colab import files
    files.download("hallucination_experiment_outputs.zip")
except ImportError:
    print("Not running in Colab — find hallucination_experiment_outputs.zip in the working directory.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
import os

for f in files_to_zip:
    if os.path.exists(f):
        print(f"✅ {f} exists")
    else:
        print(f"❌ {f} is missing")

✅ results_no_context.json exists
✅ results_rag.json exists
✅ manual_scores_no_context.json exists
✅ manual_scores_rag.json exists
✅ rate_comparison.json exists
✅ detector_results.json exists
